In [ ]:
from kalshi_python_sync import Configuration, KalshiClient
import time
from datetime import datetime

# ================== CONFIG ==================
HOST = "https://api.elections.kalshi.com/trade-api/v2"   # production

config = Configuration(host=HOST)
client = KalshiClient(config)

print("Fetching ALL open MLB / Pro Baseball markets (KXMLB series)...\n")
print(f"{'Ticker':<15} {'Title':<60} {'Yes':<8} {'No':<8} {'Volume':<12} {'Closes'}")
print("-" * 120)

markets_fetched = 0
cursor = None

while True:
    response = client.get_markets(
        limit=1000,
        cursor=cursor,
        status="open",
        series_ticker="KXMLB"          # ← THIS IS THE MAGIC LINE
    )
    
    for market in response.markets:
        markets_fetched += 1
        
        yes_price = f"${float(market.yes_ask_dollars or 0):.2f}"
        no_price  = f"${float(market.no_ask_dollars or 0):.2f}"
        
        # volume fallback (works across SDK versions)
        volume_raw = getattr(market, 'volume_fp', None) or getattr(market, 'volume', 0) or 0
        volume = f"{int(float(volume_raw)):,}"
        
        close_str = market.close_time.strftime("%b %d %H:%M")

        print(f"{market.ticker:<15} {market.title[:57]:<60} {yes_price:<8} {no_price:<8} {volume:<12} {close_str}")

    cursor = response.cursor
    if not cursor:
        break
    time.sleep(0.25)   # polite API rate limit

print(f"\n✅ Done! Found {markets_fetched} clean MLB/Pro Baseball markets")